In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass

import config as cfg

RNG_SEED = 42
N_MONTHS = 24
START_MONTH = "2024-08"

rng = np.random.default_rng(RNG_SEED)

CLIMATE_ZONE_PARAMS = {
    "arid":   {"base": 60, "amp": 40, "noise": 10},
    "low":    {"base": 80, "amp": 45, "noise": 9},
    "medium": {"base": 100, "amp": 55, "noise": 8},
    "high":   {"base": 130, "amp": 65, "noise": 9},
}

# 1. District-level climate + market indices

In [2]:
def generate_district_indices() -> pd.DataFrame:
    months = pd.period_range(START_MONTH, periods=N_MONTHS, freq="M")
    t = np.arange(N_MONTHS)

    milk_price_state = 100 + 5 * np.sin((t % 12) / 12 * 2 * np.pi) + rng.normal(0, 2, N_MONTHS)
    milk_price_state[9:12] -= 12

    feed_price_state = 100 + rng.normal(0, 3, N_MONTHS).cumsum() * 0.3
    feed_price_state[6:9] += 18

    raw_material_state = 100 + t * 0.4 + rng.normal(0, 3, N_MONTHS)

    festival_bump = np.array([15 if (m % 12) in (9, 10) else 0 for m in t])
    retail_demand_state = 100 + festival_bump + rng.normal(0, 4, N_MONTHS)

    districts = sorted(set(d for d, p, z in cfg.PLACES))
    disrupted_districts = set(rng.choice(districts, size=max(1, int(len(districts) * 0.3)), replace=False))

    rows = []
    for district in districts:
        zone = next(z for d, p, z in cfg.PLACES if d == district)
        params = CLIMATE_ZONE_PARAMS[zone]

        monsoon_curve = params["base"] + params["amp"] * np.sin((t % 12 - 3) / 12 * 2 * np.pi)
        rainfall = monsoon_curve + rng.normal(0, params["noise"], N_MONTHS)
        if rng.random() < 0.4:
            shock_start = int(rng.integers(10, 18))
            rainfall[shock_start:shock_start + 3] -= rng.uniform(20, 40)

        district_mult = rng.uniform(0.95, 1.05)
        milk_price = milk_price_state * district_mult + rng.normal(0, 1.5, N_MONTHS)
        feed_price = feed_price_state * district_mult + rng.normal(0, 1.5, N_MONTHS)
        raw_material = raw_material_state * district_mult + rng.normal(0, 1.5, N_MONTHS)
        retail_demand = retail_demand_state * district_mult + rng.normal(0, 2, N_MONTHS)

        local_disruption_flag = np.zeros(N_MONTHS, dtype=bool)
        if district in disrupted_districts:
            dstart = int(rng.integers(0, N_MONTHS - 2))
            dlen = int(rng.integers(1, 3))
            local_disruption_flag[dstart:dstart + dlen] = True
            retail_demand[dstart:dstart + dlen] -= rng.uniform(15, 30)
            raw_material[dstart:dstart + dlen] += rng.uniform(8, 15)

        for m_i, month in enumerate(months):
            rows.append({
                "district": district,
                "month": str(month),
                "rainfall_index": round(float(max(rainfall[m_i], 5.0)), 1),
                "milk_price_index": round(float(max(milk_price[m_i], 5.0)), 1),
                "poultry_feed_price_index": round(float(max(feed_price[m_i], 5.0)), 1),
                "raw_material_price_index": round(float(max(raw_material[m_i], 5.0)), 1),
                "retail_demand_index": round(float(max(retail_demand[m_i], 5.0)), 1),
                "local_disruption": bool(local_disruption_flag[m_i]),
            })

    return pd.DataFrame(rows)

# 2. Enterprises

In [3]:
@dataclass
class Enterprise:
    enterprise_id: str
    name: str
    sector: str
    district: str
    place: str
    vintage_years: float
    digital_adoption: float
    upi_adoption_start: int
    scripted_stress: bool
    performance_multiplier: float
    has_loan: bool
    loan_amount: float
    loan_tenure_months: int
    loan_start_offset: int

In [4]:
def _make_enterprise_id(place: str, used_ids: set) -> str:
    prefix = "GJ" + place[:2].upper()
    while True:
        digits = f"{rng.integers(0, 1_000_000):06d}"
        eid = prefix + digits
        if eid not in used_ids:
            used_ids.add(eid)
            return eid

In [5]:
def generate_enterprises() -> list[Enterprise]:
    total = int(rng.integers(300, 401))

    n_sectors = len(cfg.SECTORS)
    min_each = 10
    remainder = total - min_each * n_sectors
    weights = rng.dirichlet(np.ones(n_sectors) * 2)
    extra = np.floor(weights * remainder).astype(int)
    counts = extra + min_each
    counts[0] += total - counts.sum()

    used_ids = set()
    enterprises = []

    for sector, n in zip(cfg.SECTORS, counts):
        n_stress = max(1, int(n * 0.2))
        stress_flags = np.array([True] * n_stress + [False] * (n - n_stress))
        rng.shuffle(stress_flags)

        for i in range(n):
            district, place, zone = cfg.sector_district_choice(rng, sector)
            eid = _make_enterprise_id(place, used_ids)
            name = cfg.generate_enterprise_name(rng, sector)

            vintage = float(rng.gamma(shape=2.0, scale=2.5))
            vintage = min(vintage, 25.0)

            b2b_sectors = {"brick_kiln", "agri_input_retail", "auto_repair", "flour_mill", "oil_mill"}
            base_adoption = 0.55 if sector in b2b_sectors else 0.4
            digital_adoption = float(np.clip(rng.normal(base_adoption, 0.18), 0.1, 0.95))

            adoption_start = int(np.clip(
                rng.normal(6 - digital_adoption * 8 + vintage * 0.3, 3), 0, N_MONTHS - 2
            ))

            performance_multiplier = float(np.clip(rng.lognormal(mean=0.0, sigma=0.5), 0.35, 3.2))

            loan_approval_prob = np.clip(0.35 + min(vintage, 10) * 0.015 + (performance_multiplier - 1) * 0.15, 0.15, 0.9)
            has_loan = rng.random() < loan_approval_prob

            loan_amount = 0.0
            loan_tenure_months = 0
            loan_start_offset = 0
            if has_loan:
                size_scaling = np.sqrt(max(performance_multiplier, 0.4))
                loan_amount = float(rng.uniform(40000, 180000) * (1 + min(vintage, 10) * 0.03) * size_scaling)
                loan_tenure_months = int(rng.choice(
                    [6, 9, 12, 18, 24, 30, 36, 48],
                    p=[0.08, 0.10, 0.20, 0.17, 0.20, 0.10, 0.09, 0.06]
                ))
                earliest_start = -(loan_tenure_months - 3)
                latest_start = N_MONTHS - 2
                loan_start_offset = int(rng.integers(earliest_start, latest_start + 1))

            enterprises.append(Enterprise(
                enterprise_id=eid, name=name, sector=sector, district=district, place=place,
                vintage_years=round(vintage, 1), digital_adoption=round(digital_adoption, 2),
                upi_adoption_start=adoption_start, scripted_stress=bool(stress_flags[i]),
                performance_multiplier=round(performance_multiplier, 3),
                has_loan=bool(has_loan), loan_amount=round(loan_amount, 2),
                loan_tenure_months=loan_tenure_months, loan_start_offset=loan_start_offset,
            ))

    return enterprises

# 2b. Field investigator assignment

In [6]:
def assign_field_investigators(enterprises: list["Enterprise"], target_size: int = 20) -> dict:
    """Assign each enterprise to a field investigator with roughly balanced
    caseloads (~target_size enterprises each), instead of leaving FI-level
    structure to be reverse-engineered from the enterprise_id's 2-letter
    place prefix (which produced clusters as small as n=3, too noisy for a
    reliable FI-level alarm rate). Enterprises are grouped by district+place
    first so a given FI's portfolio stays geographically coherent."""
    ordered = sorted(enterprises, key=lambda e: (e.district, e.place, e.enterprise_id))
    assignment = {}
    fi_counter = 1
    current_fi = f"FI{fi_counter:03d}"
    count_in_current = 0
    for e in ordered:
        if count_in_current >= target_size:
            fi_counter += 1
            current_fi = f"FI{fi_counter:03d}"
            count_in_current = 0
        assignment[e.enterprise_id] = current_fi
        count_in_current += 1

    if count_in_current < target_size // 2 and fi_counter > 1:
        leftover_fi = current_fi
        prev_fi = f"FI{fi_counter - 1:03d}"
        for k, v in assignment.items():
            if v == leftover_fi:
                assignment[k] = prev_fi

    return assignment

# 3. Monthly financial + UPI + repayment records

In [7]:
def generate_monthly_records(enterprises: list[Enterprise], district_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    months = sorted(district_df["month"].unique())
    district_lookup = {d: sub.reset_index(drop=True) for d, sub in district_df.groupby("district")}

    for ent in enterprises:
        mkt = district_lookup[ent.district]
        base_income = cfg.SECTOR_BASE_INCOME[ent.sector] * ent.performance_multiplier
        base_income *= (1 + min(ent.vintage_years, 15) * 0.015)
        seasonality = cfg.SECTOR_SEASONALITY[ent.sector]

        if ent.has_loan and ent.loan_tenure_months > 0:
            annual_rate = float(rng.uniform(0.18, 0.28)) 
            monthly_rate = annual_rate / 12
            n = ent.loan_tenure_months
            if monthly_rate > 0:
                emi = ent.loan_amount * monthly_rate * (1 + monthly_rate) ** n / ((1 + monthly_rate) ** n - 1)
            else:
                emi = ent.loan_amount / n
        else:
            emi = 0.0
        savings_balance = rng.uniform(5000, 25000)

        drift = rng.normal(0, 0.003)
        expense_ratio = rng.uniform(0.55, 0.75)
        base_save_rate = float(rng.uniform(0.08, 0.22))
        withdrawal_shock_prob = float(rng.uniform(0.03, 0.09))
        windfall_prob = float(rng.uniform(0.02, 0.05))

        for m_i, month in enumerate(months):
            calendar_month = (pd.Period(month, freq="M").month - 1)
            season_mult = seasonality[calendar_month]
            row_mkt = mkt.iloc[m_i]

            if ent.sector == "dairy":
                shock_mult = (row_mkt["milk_price_index"] / 100) * (row_mkt["rainfall_index"] / 100) ** 0.3
            elif ent.sector == "poultry":
                shock_mult = (100 / row_mkt["poultry_feed_price_index"]) ** 1.2
            elif ent.sector in ("food_processing", "flour_mill", "oil_mill", "textile_weaving"):
                shock_mult = (100 / row_mkt["raw_material_price_index"]) ** 0.5
            elif ent.sector in ("handicrafts", "rural_retail", "tea_stall_eatery"):
                shock_mult = (row_mkt["retail_demand_index"] / 100)
            elif ent.sector == "brick_kiln":
                shock_mult = (row_mkt["rainfall_index"] / 100) ** -0.4
            elif ent.sector == "agri_input_retail":
                shock_mult = (row_mkt["rainfall_index"] / 100) ** 0.3
            else:
                shock_mult = (row_mkt["retail_demand_index"] / 100) ** 0.5

            if row_mkt["local_disruption"]:
                shock_mult *= rng.uniform(0.75, 0.9)

            scripted_hit = 1.0
            if ent.scripted_stress and 10 <= m_i <= 15:
                stage = m_i - 10
                scripted_hit = 1.0 - [0.05, 0.15, 0.30, 0.35, 0.25, 0.12][stage]

            idio_noise = rng.normal(1.0, 0.06)
            income = base_income * season_mult * shock_mult * scripted_hit * idio_noise * (1 + drift * m_i)
            income = max(income, base_income * 0.15)

            expenses = income * expense_ratio * rng.uniform(0.92, 1.08)

            months_since_adoption = m_i - ent.upi_adoption_start
            if months_since_adoption < 0:
                effective_adoption = 0.02
            else:
                ramp = min(1.0, months_since_adoption / 4)
                effective_adoption = ent.digital_adoption * ramp

            upi_txn_volume = income * effective_adoption * rng.uniform(0.85, 1.15)
            upi_txn_count = max(0, int((income / 900) * effective_adoption * rng.uniform(0.8, 1.2)))
            if upi_txn_count == 0 and upi_txn_volume > 1:
                upi_txn_count = 1

            cash_before_emi = income - expenses

            loan_active = False
            if ent.has_loan:
                loan_end = ent.loan_start_offset + ent.loan_tenure_months
                loan_active = ent.loan_start_offset <= m_i < loan_end

            net_cash_flow = cash_before_emi - (emi if loan_active else 0)

            if not ent.has_loan:
                repayment_status = "no_loan"
            elif m_i < ent.loan_start_offset:
                repayment_status = "no_loan"       
            elif not loan_active:
                repayment_status = "loan_closed"   
            else:
                covered = cash_before_emi + min(savings_balance, emi) * 0.3
                if covered < emi * 0.7:
                    repayment_status = "missed"
                elif covered < emi * 1.15:
                    repayment_status = "late"
                else:
                    repayment_status = "on_time"

            savings_contribution = net_cash_flow * base_save_rate * rng.uniform(0.6, 1.4)
            shock = 0.0
            if rng.random() < withdrawal_shock_prob:
                # emergency draw-down: medical bill, urgent repair, festival expense
                shock = -savings_balance * float(rng.uniform(0.15, 0.5))
            elif rng.random() < windfall_prob:
                # external cash injection not visible in income/expenses: gift,
                # family support, small asset sale
                shock = float(rng.uniform(2000, 15000))
            savings_balance = max(0, savings_balance + savings_contribution + shock)

            underreport_prob = 0.08 * (1 - ent.digital_adoption)
            data_complete = rng.random() > underreport_prob

            rows.append({
                "enterprise_id": ent.enterprise_id,
                "month": month,
                "income": round(income, 2),
                "expenses": round(expenses, 2),
                "net_cash_flow": round(net_cash_flow, 2),
                "savings_balance": round(savings_balance, 2),
                "has_loan": loan_active,
                "emi_due": round(emi, 2) if loan_active else 0.0,
                "repayment_status": repayment_status,
                "upi_inflow_txn_count": upi_txn_count,
                "upi_inflow_txn_volume": round(upi_txn_volume, 2),
                "data_complete": bool(data_complete),
            })

    return pd.DataFrame(rows)

In [9]:
def main():
    district_df = generate_district_indices()
    enterprises = generate_enterprises()
    fi_assignment = assign_field_investigators(enterprises)

    ent_df = pd.DataFrame([{
        "enterprise_id": e.enterprise_id,
        "name": e.name,
        "sector": e.sector,
        "district": e.district,
        "place": e.place,
        "field_investigator_id": fi_assignment[e.enterprise_id],
        "vintage_years": e.vintage_years,
        "digital_adoption": e.digital_adoption,
        "upi_adoption_start_month": e.upi_adoption_start,
        "loan_taken_in_dataset": e.has_loan,
        "loan_amount": e.loan_amount,
        "loan_tenure_months": e.loan_tenure_months,
        "loan_start_month_offset": e.loan_start_offset,
    } for e in enterprises])

    ground_truth_df = pd.DataFrame([{
        "enterprise_id": e.enterprise_id,
        "scripted_stress": e.scripted_stress,
        "performance_multiplier": e.performance_multiplier,
    } for e in enterprises])

    records_df = generate_monthly_records(enterprises, district_df)
    records_df = records_df.merge(
        pd.DataFrame([{"enterprise_id": e.enterprise_id, "field_investigator_id": fi_assignment[e.enterprise_id]}
                      for e in enterprises]),
        on="enterprise_id", how="left",
    )

    district_df.to_csv("data/district_indices.csv", index=False)
    ent_df.to_csv("data/enterprises.csv", index=False)
    ground_truth_df.to_csv("data/enterprise_ground_truth.csv", index=False)
    records_df.to_csv("data/monthly_records.csv", index=False)

    print(f"Total enterprises: {len(ent_df)}")
    print(ent_df["sector"].value_counts())
    print(f"\nDistricts covered: {ent_df['district'].nunique()}")
    print(f"Monthly records: {len(records_df)}")
    print(f"Scripted stress enterprises: {ground_truth_df['scripted_stress'].sum()}")
    print(f"Enterprises with a loan schedule: {ent_df['loan_taken_in_dataset'].sum()}")
    print(f"\nPerformance multiplier distribution:\n{ground_truth_df['performance_multiplier'].describe()}")
    print(f"\nField investigators: {ent_df['field_investigator_id'].nunique()}")
    print(f"Caseload per FI:\n{ent_df['field_investigator_id'].value_counts().describe()}")
    print(f"\nMax monthly income in dataset: {records_df['income'].max():,.0f}")
    print(f"Enterprises-months with income > 100,000: {(records_df['income']>100000).sum()}")


if __name__ == "__main__":
    main()

Total enterprises: 372
sector
tea_stall_eatery     58
food_processing      51
agri_input_retail    40
auto_repair          32
oil_mill             31
handicrafts          28
poultry              27
dairy                25
brick_kiln           23
flour_mill           22
rural_retail         18
textile_weaving      17
Name: count, dtype: int64

Districts covered: 28
Monthly records: 8928
Scripted stress enterprises: 70
Enterprises with a loan schedule: 159

Performance multiplier distribution:
count    372.000000
mean       1.109761
std        0.565068
min        0.350000
25%        0.671500
50%        0.996000
75%        1.368000
max        3.200000
Name: performance_multiplier, dtype: float64

Field investigators: 19
Caseload per FI:
count    19.000000
mean     19.578947
std       1.835326
min      12.000000
25%      20.000000
50%      20.000000
75%      20.000000
max      20.000000
Name: count, dtype: float64

Max monthly income in dataset: 187,345
Enterprises-months with income > 100